# Cloudways API quickstart

Explore the Cloudways API v2 from Python. Read-only until the last (opt-in) section.

**Before you start:** create an Access Token (Profile -> API Integration). A **Read-Only** token is enough for everything except the final cell. Never commit it.

In [ ]:
import getpass, json, os, urllib.request, urllib.error

BASE = 'https://api.cloudways.com/api/v2'  # (verify) derived from v1->v2 migration note
os.environ['CLOUDWAYS_TOKEN'] = os.environ.get('CLOUDWAYS_TOKEN') or getpass.getpass('Access Token: ')

def api(method, path, body=None):
    req = urllib.request.Request(BASE + path, method=method,
        data=json.dumps(body).encode() if body is not None else None,
        headers={'Authorization': 'Bearer ' + os.environ['CLOUDWAYS_TOKEN'],
                 'Accept': 'application/json', 'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return json.loads(r.read() or b'{}')
    except urllib.error.HTTPError as e:
        raise SystemExit(f'HTTP {e.code}: {e.read().decode()[:500]}')

## 1. List servers and apps

In [ ]:
data = api('GET', '/server')
for s in data.get('servers', []):
    print(f"[{s.get('id')}] {s.get('label')}  {s.get('cloud')}/{s.get('region')}  {s.get('public_ip')}")
    for a in s.get('apps', []):
        print(f"    app {a.get('id')}: {a.get('label')} ({a.get('application')})")

## 2. Inspect raw JSON
Open the Playground side by side and compare shapes. Field names can differ from what the scripts assume.

In [ ]:
print(json.dumps(data, indent=2)[:2000])

## 3. Discover valid launch options
The docs group reference data under **Lists** (providers, regions, sizes, apps). Find the exact paths in the [docs](https://developers.cloudways.com/docs), then query them here instead of guessing values.

In [ ]:
# Example (verify the path in the docs):
# print(json.dumps(api('GET', '/providers'), indent=2)[:1500])

## 4. (Optional) Launch a server
**This can incur charges and acts on your real account.** Needs a Full or Limited token that allows `POST /server`. Fill the fields with values from step 3, then set `CONFIRM = True`.

In [ ]:
CONFIRM = False
payload = {
    'cloud': '', 'region': '', 'instance_type': '', 'application': 'wordpress',
    'app_version': '', 'server_label': 'nb-test', 'app_label': 'nb-app', 'project_name': ''
}
print(json.dumps(payload, indent=2))
if CONFIRM:
    print(api('POST', '/server', payload))  # returns an operation id; poll /operation/<id>